### 모델 경량화(Quantization)
- 파일 : n2-1_quantization.ipynb
- 역할 : FP16 변환과 INT8 양자화를 적용하고, 크기와 추론 결과의 변화를 비교합니다.
- 기능 :
    1. FP32 → FP16 변환  
    2. 공통 데이터셋 Imagenette 준비  
    3. CalibrationDataReader 구현  
    4. quantize_static 실행  
    5. FP32·FP16·INT8 Top-1 정확도 비교  


>> [준비] 패키지와 ONNX 파일 준비 (노트북 전용 셀)

In [ ]:
#- 경량화 실습 패키지 설치
!pip install -q onnx onnxruntime onnxconverter-common "protobuf<6"

#- ONNX 변환과 검증 노트북의 산출물(mobilenet_v2.onnx, resnet18.onnx)을 이 세션에 준비
import os
import torch
from torchvision import models

for name, build, w in [
    ("mobilenet_v2", models.mobilenet_v2, models.MobileNet_V2_Weights.IMAGENET1K_V1),
    ("resnet18",     models.resnet18,     models.ResNet18_Weights.IMAGENET1K_V1),
]:
    if not os.path.exists(f"{name}.onnx"):
        m = build(weights=w).eval()
        torch.onnx.export(m, torch.randn(1, 3, 224, 224), f"{name}.onnx",
                          opset_version=13, input_names=["input"],
                          output_names=["output"], dynamo=False)
print(sorted(f for f in os.listdir(".") if f.endswith(".onnx")))

# 확인 포인트 ------------------------------------------
# - mobilenet_v2.onnx, resnet18.onnx 두 파일 출력 체크

>> [코드 4-1] FP32 → FP16 변환 (n2-1_quantization.ipynb, STEP 1)
<hr>

In [ ]:
#- 모듈로딩
import onnx, os
from onnxconverter_common import float16

#- 모델 상수: 경량화 대상 두 모델 이름
MODEL_NAMES = ["mobilenet_v2", "resnet18"]

#- 두 모델을 FP16으로 변환하고 크기 비교
for name in MODEL_NAMES:
    # FP32 그래프 로딩
    m = onnx.load(f"{name}.onnx")
    # 가중치·활성값 자료형을 FP16으로
    m_fp16 = float16.convert_float_to_float16(m)
    onnx.save(m_fp16, f"{name}_fp16.onnx")
    # 변환 전후 파일 크기(MB)
    s32 = os.path.getsize(f"{name}.onnx") / 1e6
    s16 = os.path.getsize(f"{name}_fp16.onnx") / 1e6
    print(f"{name:14s} FP32 {s32:5.1f} MB → FP16 {s16:5.1f} MB ({s16/s32*100:.0f}%)")

# 확인 포인트 ------------------------------------------
# - FP32 대비 약 50% 크기 출력 체크
# - truncated 경고는 정상 (FP16 표현 범위 밖의 작은 값)

**실행 결과 예시**
```
mobilenet_v2   FP32  14.0 MB → FP16   7.0 MB (50%)
resnet18       FP32  46.8 MB → FP16  23.4 MB (50%)
```

>> [코드 4-2] 공통 데이터셋 Imagenette 준비 (STEP 2)
<hr>

In [ ]:
#- 모듈로딩
import urllib.request, tarfile, glob, os

#- 데이터셋 상수: 압축 파일과 풀린 폴더 (경량화·NPU 양자화 공용)
DS_FILE = "imagenette2-160.tgz"
DS_DIR = "imagenette2-160"
TRAIN_DIR = f"{DS_DIR}/train"

#- Imagenette 다운로드와 압축 해제 (한 번만)
URL = f"https://s3.amazonaws.com/fast-ai-imageclas/{DS_FILE}"
if not os.path.exists(DS_DIR):
    # 160px 판 내려받기 (약 100MB)
    urllib.request.urlretrieve(URL, "imagenette.tgz")
    with tarfile.open("imagenette.tgz") as t:
        t.extractall(".", filter="data")

#- 학습 이미지 목록
train_files = sorted(glob.glob(f"{TRAIN_DIR}/*/*.JPEG"))
print(f"train 이미지 수: {len(train_files):,}")
# 확인 포인트 ------------------------------------------
# - train 이미지 수 출력 체크

**실행 결과 예시**
```
train 이미지 수: 9,469
```

>> [코드 4-3] CalibrationDataReader 구현 (STEP 3)
<hr>

In [ ]:
#- 모듈로딩
import numpy as np
from PIL import Image
from onnxruntime.quantization import CalibrationDataReader

#- FP32 파이프라인과 동일한 전처리 상수
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

#- 이미지 1장을 모델 입력 형식으로
def load_one(path):
    # RGB 변환과 224×224 크기 맞춤
    img = Image.open(path).convert("RGB").resize((224, 224))
    # /255 후 mean/std 정규화
    x = (np.asarray(img).astype(np.float32)/255.0 - MEAN) / STD
    # NCHW로 바꾸고 배치 축 추가
    return x.transpose(2, 0, 1)[None]

#- Calibration 데이터 공급자: 등간격 100장을 순서대로 제공
class ImagenetteCalibReader(CalibrationDataReader):
    def __init__(self, files, n=100):
        # 전체 목록에서 등간격으로 n장
        step = max(1, len(files) // n)
        self.files = files[::step][:n]      # 전체 목록에서 등간격으로 100장 선택
        self.it = None

    #- 양자화 도구가 반복 호출: 다음 입력 1개, 소진되면 None
    def get_next(self):
        if self.it is None:
            self.it = iter({"input": load_one(f)} for f in self.files)
        return next(self.it, None)

# 확인 포인트 -------------------------------------------------------------------------
# - 함수·클래스 정의 셀: 출력 없음

>> [코드 4-4] quantize_static 실행 (STEP 4)
<hr>

In [ ]:
#- 모듈로딩
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat

#- 두 모델을 INT8(QDQ)로 정적 양자화
for name in MODEL_NAMES:
    quantize_static(
        model_input=f"{name}.onnx",
        model_output=f"{name}_int8.onnx",
        calibration_data_reader=ImagenetteCalibReader(train_files),
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QUInt8,   # 활성값: 부호 없는 8비트 정수
        weight_type=QuantType.QInt8,        # 가중치: 부호 있는 8비트 정수
    )

    s32 = os.path.getsize(f"{name}.onnx") / 1e6
    s8  = os.path.getsize(f"{name}_int8.onnx") / 1e6
    print(f"{name:14s} FP32 {s32:5.1f} MB → INT8 {s8:5.1f} MB ({s8/s32*100:.0f}%)")

# 확인 포인트 -------------------------------------------------------------------------
# - 두 모델 모두 약 25% 크기로 감소 체크 (Calibration 100장 포함 수 분 소요)

>> [코드 4-5] FP32·FP16·INT8 Top-1 정확도 비교 (STEP 5)
<hr>

In [ ]:
#- 모듈로딩
import onnxruntime as ort

#- 상수: 비교할 세 정밀도 접미사
TAGS = ["", "_fp16", "_int8"]

#- Imagenette 10클래스와 ImageNet 인덱스 매핑, 검증 목록
IMAGENET_IDX = [0, 217, 482, 491, 497, 566, 569, 571, 574, 701]
WNIDS = sorted(os.listdir("imagenette2-160/val"))
val_files = sorted(glob.glob(f"{DS_DIR}/val/*/*.JPEG"))[::10]

#- Top-1 평가 (Imagenette 10클래스 안에서)
def evaluate(model_path, files):

    sess     = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
    in_dtype = np.float16 if "fp16" in model_path else np.float32
    correct = 0

    for f in files:
        # 폴더명(wnid)에서 정답 인덱스
        gt = IMAGENET_IDX[WNIDS.index(f.split("/")[-2])]
        # 추론: 로짓 1,000개
        out = sess.run(None, {"input": load_one(f).astype(in_dtype)})[0][0]
        # 10클래스 점수만 비교해 Top-1
        pred = IMAGENET_IDX[int(np.argmax(out[IMAGENET_IDX]))]
        correct += (pred == gt)
    return correct / len(files)

#- 세 정밀도의 Top-1 비교
for tag in TAGS:
    acc = evaluate(f"mobilenet_v2{tag}.onnx", val_files)
    print(f"MobileNetV2 {tag or 'FP32':6s} Top-1: {acc*100:5.1f}%")

# 확인 포인트 --------------------------------------------------------------------------
# - 세 정밀도의 Top-1 출력 체크, INT8 하락 폭 확인